In [2]:
import pandas as pd
from os import path

# 'Name', 'Vessel Sanction Indicators', 'IMO', 'Flag', 'LLI Vessel Type', 'Vessel Type and Size', 'Built', 'Status', 'Beneficial Owner',
# 'Beneficial Owner Country/Region', 'Beneficial Owner Sanction Indicators', 'Beneficial Owner - Shareholders & UBO Sanctions',
# 'Commercial Operator', 'Commercial Operator Country/Region', 'Commercial Operator Sanction Indicators', 'Commercial Operator - Shareholders & UBO Sanctions',
# 'Registered Owner', 'Registered Owner Country/Region', 'Registered Owner Sanction Indicators', 'Registered Owner - Shareholders & UBO Sanctions', 'Technical Manager',
# 'Technical Manager Country/Region', 'Technical Manager Sanction Indicators', 'Technical Manager - Shareholders & UBO Sanctions', 'ISM Manager',
# 'ISM Manager Country/Region', 'ISM Manager Sanction Indicators', 'ISM Manager - Shareholders & UBO Sanctions', 'Third Party Operator', 'Third Party Operator Country/Region',
# 'Third Party Operator Sanction Indicators', 'Third Party Operator - Shareholders & UBO Sanctions', 'Length Overall (m)', 'Depth (m)', 'Maximum Draught (m)', 'Freeboard (m)'
vessels = pd.read_csv(path.join("data", "Vessels.csv"), sep=";")

# 'subjectid', 'timestamp', 'position', 'cog', 'sog', 'trueheading', 'navstatus', 'lng', 'lat'
radar_pos = pd.read_csv(path.join("data","AIS_RADAR_streams202260306", "radar_positionrecord_202603061623.csv"))

# 'vesselid', 'timestamp', 'position', 'cog', 'sog', 'trueheading', 'navstatus', 'lng', 'lat'
ais_pos = pd.read_csv(path.join("data","AIS_RADAR_streams202260306", "vessel_annotatedpositionrecord_202603061619.csv"))

print(f"vessels: {len(vessels)}, radar_pos: {len(radar_pos)}, vessel_ann: {len(ais_pos)}")

radar_pos[["lng", "lat"]] = radar_pos["position"].str.extract(r"POINT \(([^ ]+) ([^ ]+)\)").astype(float)
ais_pos[["lng", "lat"]] = ais_pos["position"].str.extract(r"POINT \(([^ ]+) ([^ ]+)\)").astype(float)

print("radar_pos", radar_pos["subjectid"].nunique(), radar_pos["timestamp"].min(), "->", radar_pos["timestamp"].max())
print("vessel_ann", ais_pos["vesselid"].nunique(), ais_pos["timestamp"].min(), "->", ais_pos["timestamp"].max())

vessels: 6151, radar_pos: 335382, vessel_ann: 3240214
radar_pos 2934 2026-03-06 13:33:00.388 -> 2026-03-06 14:33:59.896
vessel_ann 194932 2026-03-06 13:33:00.000 -> 2026-03-06 14:33:59.895


In [23]:
import geopandas as gpd
from shapely.geometry import LineString, Point

def build_tracks(df, id_col):
    def to_geom(x):
        coords = x.values
        if len(coords) == 1:
            return Point(coords[0])
        return LineString(coords)

    tracks = (
        df.sort_values("timestamp")
        .groupby(id_col)[["lng", "lat"]]
        .apply(to_geom)
    )
    return gpd.GeoDataFrame({"geometry": tracks}, crs="EPSG:4326")

radar_tracks = build_tracks(radar_pos, "subjectid")
ais_tracks = build_tracks(ais_pos, "vesselid")

print(f"radar tracks: {len(radar_tracks)}, ais tracks: {len(ais_tracks)}")

radar tracks: 2934, ais tracks: 194932


In [24]:
HAUSDORFF_THRESHOLD = 0.01  # degrees (~1km)

ais_sindex = ais_tracks.sindex
matches = []

for subjectid, radar_geom in radar_tracks.geometry.items():
    candidate_idx = list(ais_sindex.intersection(radar_geom.bounds))
    if not candidate_idx:
        continue

    candidates = ais_tracks.iloc[candidate_idx]
    distances = candidates.geometry.apply(lambda g: radar_geom.hausdorff_distance(g))

    for vesselid, dist in distances[distances <= HAUSDORFF_THRESHOLD].items():
        matches.append({
            "subjectid": subjectid,
            "vesselid": vesselid,
            "hausdorff_distance": dist,
        })

results = pd.DataFrame(matches).sort_values("hausdorff_distance")
print(f"Found {len(results)} matching track pairs across {results['vesselid'].nunique()} AIS vessels")

pd.set_option("display.max_rows", None)
results

Found 124 matching track pairs across 55 AIS vessels


,subjectid,vesselid,hausdorff_distance
5,153162546,3967854373,0.000433
2,153127015,27716836806,0.000470
118,153231219,3967848979,0.000578
25,153219235,46738644503,0.001112
52,153225164,3967948521,0.001251
54,153225238,3967948521,0.001265
70,153227712,3967820749,0.001313
55,153225398,3967948521,0.001332
21,153213309,3967948521,0.001357
68,153227233,3968428716,0.001403


In [30]:
import json
from shapely.geometry import MultiLineString, MultiPoint

# vesselid = 3968029872
# subjectids = [153183711, 153185216, 153183240, 153186863, 153185561]

vesselid = 3967854373
subjectids = [153162546]

valid_sids = [sid for sid in subjectids if sid in radar_tracks.index]
geoms = [radar_tracks.loc[sid, "geometry"] for sid in valid_sids]

lines = [g for g in geoms if g.geom_type == "LineString"]
points = [g for g in geoms if g.geom_type == "Point"]

combined_radar = MultiLineString(lines) if lines else MultiPoint(points)

features = [
    {"type": "Feature", "properties": {"ids": valid_sids, "source": "radar"}, "geometry": combined_radar.__geo_interface__},
    {"type": "Feature", "properties": {"id": vesselid, "source": "ais"}, "geometry": ais_tracks.loc[vesselid].geometry.__geo_interface__},
]

with open("track.geojson", "w") as f:
    json.dump({"type": "FeatureCollection", "features": features}, f)

print(f"Saved track.geojson — radar: {combined_radar.geom_type}, {len(lines)} lines + {len(points)} points")

Saved track.geojson — radar: MultiLineString, 1 lines + 0 points
